In [20]:
# Librerías para manipulación, visualización y carga portable de datos.
import pandas as pd
# import numpy as np 
# import matplotlib.pyplot as plt
# import seaborn as sns
import os

# Usamos una ruta para Kaggle y otra para ejecución local en VS Code.
kaggle_path = '/kaggle/input/california-housing-prices/housing.csv'
local_path = 'housing.csv'

# Cargamos el dataset desde la ruta que exista en el entorno actual.
if os.path.exists(kaggle_path):
    df = pd.read_csv(kaggle_path)
    print("Cargado desde el entorno de Kaggle")
elif os.path.exists(local_path):
    df = pd.read_csv(local_path)
    print("Cargado desde el entorno local (VS Code)")
else:
    raise FileNotFoundError("No se encontro el archivo 'housing.csv' en ninguna de las rutas.")

# Vista inicial: permite comprobar que las columnas se cargaron correctamente.
df.head()

Cargado desde el entorno local (VS Code)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


In [22]:
print("\nValores nulos por columna:")
print(df.isnull().sum())


Valores nulos por columna:
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64


In [23]:
df.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


In [24]:
from sklearn.model_selection import train_test_split

# Separamos features (X) del objetivo (y)
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]

# Split SIN trampas: 80% train, 20% test, con semilla fija para reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape[0]} filas")
print(f"Test: {X_test.shape[0]} filas")


Train: 16512 filas
Test: 4128 filas


In [25]:
# Calcular la mediana SOLO con train
mediana_bedrooms = X_train["total_bedrooms"].median()
print(f"Mediana de total_bedrooms (train): {mediana_bedrooms}")

# Rellenamos los nulos en ambos conjuntos con esa mediana
X_train["total_bedrooms"] = X_train["total_bedrooms"].fillna(mediana_bedrooms)
X_test["total_bedrooms"] = X_test["total_bedrooms"].fillna(mediana_bedrooms)

# Verificamos que no quedan nulos
print("Nulos en train: ", X_train["total_bedrooms"].isnull().sum())
print("Nulos en train: ", X_train["total_bedrooms"].isnull().sum())

Mediana de total_bedrooms (train): 437.0
Nulos en train:  0
Nulos en train:  0


In [26]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# fit SOLO con train
encoder.fit(X_train[["ocean_proximity"]])

def aplicar_encoding(X, encoder):
    codificado = pd.DataFrame(
        encoder.transform(X[["ocean_proximity"]]),
        columns=encoder.get_feature_names_out(["ocean_proximity"]),
        index=X.index,
    )
    return pd.concat([X.drop(columns=["ocean_proximity"]), codificado], axis=1)

X_train = aplicar_encoding(X_train, encoder)
X_test = aplicar_encoding(X_test, encoder)

print(X_train.columns.tolist())
print(X_train.shape, X_test.shape)

['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'ocean_proximity_<1H OCEAN', 'ocean_proximity_INLAND', 'ocean_proximity_ISLAND', 'ocean_proximity_NEAR BAY', 'ocean_proximity_NEAR OCEAN']
(16512, 13) (4128, 13)


In [27]:
def agregar_features(X):
    X = X.copy()
    X["rooms_per_household"] = X["total_rooms"] / X["households"]
    X["bedrooms_per_room"] = X["total_bedrooms"] / X["total_rooms"]
    X["population_per_household"] = X["population"] / X["households"]
    return X

X_train = agregar_features(X_train)
X_test = agregar_features(X_test)

print(X_train.shape, X_test.shape)
print(X_train[["rooms_per_household", "bedrooms_per_room", "population_per_household"]].describe())

(16512, 16) (4128, 16)
       rooms_per_household  bedrooms_per_room  population_per_household
count         16512.000000       16512.000000              16512.000000
mean              5.435235           0.212858                  3.096961
std               2.387375           0.057995                 11.578744
min               0.888889           0.100000                  0.692308
25%               4.452055           0.175178                  2.428799
50%               5.235874           0.202808                  2.817240
75%               6.061037           0.239501                  3.280000
max             141.909091           1.000000               1243.333333
